In [1]:
import os
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import time

In [2]:
# current_dir = os.path.dirname(os.path.abspath(__file__))

current_dir = os.getcwd() 
file_path = os.path.join(current_dir, "books", "odyssey_small.txt")
persistent_directory = os.path.join(current_dir, "db", "chroma_db1")
print(file_path)
print(persistent_directory)

/home/kazi/Works/Projects/lang-chain/rag/books/odyssey_small.txt
/home/kazi/Works/Projects/lang-chain/rag/db/chroma_db1


In [3]:
if not os.path.exists(persistent_directory):
    print("Initializing vector store...")
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"The file {file_path} does not exist. Please check the path."
        )
    loader = TextLoader(file_path)
    documents = loader.load()
    # Split the document into chunks
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    # text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    docs = text_splitter.split_documents(documents)
    print("\n--- Document Chunks Information ---")
    print(f"Number of document chunks: {len(docs)}")
    print(f"Sample chunk:\n{docs[0].page_content}\n")

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    ) 
    db = Chroma.from_documents(docs, embeddings, persist_directory=persistent_directory) 
else:
    print("Vector store already exists")

Created a chunk of size 1141, which is longer than the specified 1000
Created a chunk of size 2086, which is longer than the specified 1000


Initializing vector store...

--- Document Chunks Information ---
Number of document chunks: 8
Sample chunk:
﻿This translation is intended to supplement a work entitled “The
Authoress of the Odyssey”, which I published in 1897. I could not give
the whole “Odyssey” in that book without making it unwieldy, I
therefore epitomised my translation, which was already completed and
which I now publish in full.

I shall not here argue the two main points dealt with in the work just
mentioned; I have nothing either to add to, or to withdraw from, what I
have there written. The points in question are:

(1) that the “Odyssey” was written entirely at, and drawn entirely
from, the place now called Trapani on the West Coast of Sicily, alike
as regards the Phaeacian and the Ithaca scenes; while the voyages of
Ulysses, when once he is within easy reach of Sicily, solve themselves
into a periplus of the island, practically from Trapani back to
Trapani, via the Lipari islands, the Straits of Messina, and

In [4]:
query = "Who is Odysseus' wife?"

# Retrieve relevant documents based on the query
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.1},
)
relevant_docs = retriever.invoke(query)

for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")
    if doc.metadata:
        print(f"Source: {doc.metadata.get('source', 'Unknown')}\n")

Document 1:
Now all the rest, as many as fled from sheer destruction, were at
    home, and had escaped both war and sea, but Odysseus only, craving
    for his wife and for his homeward path, the lady nymph Calypso
    held, that fair goddess, in her hollow caves, longing to have him
    for her lord. But when now the year had come in the courses of the
    seasons, wherein the gods had ordained that he should return home
    to Ithaca, not even there was he quit of labours, not even among
    his own; but all the gods had pity on him save Poseidon, who raged
    continually against godlike Odysseus, till he came to his own
    country. Howbeit Poseidon had now departed for the distant
    Ethiopians, the Ethiopians that are sundered in twain, the
    uttermost of men, abiding some where Hyperion sinks and some where
    he rises. There he looked to receive his hecatomb of bulls and
    rams, there he made merry sitting at the feast, but the other gods
    were gathered in the halls o